In [8]:
import warnings
warnings.filterwarnings('ignore')
import json
import subprocess
import sys
import time
from pathlib import Path
from synthesis.llm_client import LLMClient
from configs.config_loader import Config
from synthesis.data_utils import (
    save_generated_file, load_json, extract_code_block,
    verify_python_syntax, verify_json_structure,
    parse_ref_fields, topological_sort, validate_foreign_keys,
    calculate_coverage_statistics, format_validation_report,
    sample_db,
)
notebook_config = Config("configs/app_notebook.yaml")


DOMAIN = "admin"
llm_model_name = "claude-sonnet-4-6"
llm = LLMClient(llm_model_name, new_config=notebook_config)

# 路径配置：生成代码和数据分离存放
PROMPTS_DIR       = Path("prompts") / DOMAIN
GENERATED_DATA_DIR = Path("bench") / "data_dir/domains" / DOMAIN
GENERATED_CODE_DIR = Path("bench") / "domains" / DOMAIN
BASE_DATA_DIR      = Path("bench") / "data_dir/domains" / "base"
BASE_CODE_DIR      = Path("bench") / "domains" / "base"
GENERATED_DATA_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_CODE_DIR.mkdir(parents=True, exist_ok=True)

print("✓ 初始化完成")
print(f"LLM: {llm.llm.model_name}")
print(f"Prompts 目录: {PROMPTS_DIR}")
print(f"生成数据目录：{GENERATED_DATA_DIR}")
print(f"生成代码目录：{GENERATED_CODE_DIR}")
print(f"BASE 数据目录：{BASE_DATA_DIR}")

✓ 初始化完成
LLM: claude-sonnet-4-6
Prompts 目录: prompts/admin
生成数据目录：bench/data_dir/domains/admin
生成代码目录：bench/domains/admin
BASE 数据目录：bench/data_dir/domains/base


### 0. 加载 BASE 上下文

In [2]:
# 加载 BASE 域生成结果（作为 ADMIN 域的基础上下文）
with open(BASE_DATA_DIR / "schema.json", 'r', encoding='utf-8') as f:
    base_schema = f.read()
with open(BASE_CODE_DIR / "models.py", 'r', encoding='utf-8') as f:
    base_models = f.read()
with open(BASE_DATA_DIR / "db.json", 'r', encoding='utf-8') as f:
    base_data_raw = f.read()
with open(BASE_CODE_DIR / "gen_data.py", 'r', encoding='utf-8') as f:
    base_gen_data_code = f.read()

base_data = json.loads(base_data_raw)
print("加载完成")
print(f"BASE 表数量：{len(base_data)}")

加载完成
BASE 表数量：16


### 1. 生成 ADMIN Schema

In [10]:
with open(PROMPTS_DIR / "1_data_schema.txt", 'r', encoding='utf-8') as f:
    schema_prompt = f.read()

schema_prompt = schema_prompt.replace("{base_schema}", base_schema)
# print(schema_prompt)


print(f"正在生成 {DOMAIN} Schema...")
print("-" * 60)
schema_response = ""
for chunk in llm.llm.stream(schema_prompt):
    content = chunk.content
    print(content, end="", flush=True)
    schema_response += content
    
print(f"\n{'-' * 60}")

schema_text = extract_code_block(schema_response)
is_json, schema_data = verify_json_structure(schema_text)
if is_json:
    print(f"✓ JSON 结构验证通过，共 {len(schema_data)} 张表：")
    for t in schema_data.keys():
        print(f"  - {t}")
else:
    print(f"✗ JSON 验证失败：{schema_data}")

schema_path = GENERATED_DATA_DIR / "schema.json"
save_generated_file(str(schema_path), schema_text)
print(f"\n✓ Schema 已保存到: {schema_path}")

正在生成 admin Schema...
------------------------------------------------------------
```json
{
  "campus": {
    "CPS001": {
      "campus_ref": "CPS001",
      "campus_name": "...",
      "campus_code": "...",
      "campus_type": "headquarters",
      "campus_status": "active",
      "address": "...",
      "city": "...",
      "country": "CN",
      "geo_lat": 0.0,
      "geo_lng": 0.0,
      "map_file_ref": "...",
      "description": "...",
      "created_at": "2025-01-15 09:00:00",
      "updated_at": "2025-01-15 09:00:00"
    }
  },

  "building": {
    "BLD001": {
      "building_ref": "BLD001",
      "campus_ref": "CPS001",
      "building_name": "...",
      "building_code": "...",
      "floor_count": 10,
      "building_status": "active",
      "description": "...",
      "created_at": "2025-01-15 09:00:00",
      "updated_at": "2025-01-15 09:00:00"
    }
  },

  "floor": {
    "FLR001": {
      "floor_ref": "FLR001",
      "building_ref": "BLD001",
      "floor_number": 1,
  

### 2. 生成 ADMIN Data Models

In [11]:
with open(GENERATED_DATA_DIR / "schema.json", 'r', encoding='utf-8') as f:
    domain_schema = f.read()

with open(PROMPTS_DIR / "2_data_models.txt", 'r', encoding='utf-8') as f:
    models_prompt = f.read()

models_prompt = models_prompt.replace("{base_models}", base_models)
models_prompt = models_prompt.replace("{domain_schema}", domain_schema)

print(f"正在生成 {DOMAIN} Data Models...")
print("-" * 60)
models_response = ""
for chunk in llm.llm.stream(models_prompt):
    content = chunk.content
    models_response += content
    print(content, end="", flush=True)
print(f"\n{'-' * 60}")

models_code = extract_code_block(models_response)
is_valid, err = verify_python_syntax(models_code)
if is_valid:
    print("✓ Python 语法验证通过")
else:
    print(f"✗ Python 语法错误: {err}")

models_path = GENERATED_CODE_DIR / "models.py"
save_generated_file(str(models_path), models_code)
print(f"✓ Models 已保存到: {models_path}")

正在生成 admin Data Models...
------------------------------------------------------------
```python
from typing import Any, Dict, List, Literal, Optional

from pydantic import BaseModel, Field

from bench.environment.db import DB
from generated.domains.BASE.models import OfficeDB


# --- Enum Literal types for ADMIN domain ---

CampusType = Literal["headquarters", "branch", "r&d_center", "manufacturing", "warehouse", "other"]
CampusStatus = Literal["active", "inactive", "closed", "under_construction"]

BuildingStatus = Literal["active", "inactive", "under_renovation", "demolished"]

FloorStatus = Literal["active", "inactive", "under_renovation"]

ZoneType = Literal["office", "meeting", "lounge", "storage", "cafeteria", "gym", "parking", "lobby", "server_room", "other"]
ZoneStatus = Literal["active", "inactive", "under_renovation"]

FacilityType = Literal["meeting_room", "conference_hall", "phone_booth", "gym", "lounge", "prayer_room", "nursing_room", "recreation_room", "other"]
FacilitySt

### 3.Schema/Model一致性检查

In [12]:
validation_prompt = f"""Schema:
{domain_schema}

Data model:
{models_code}

请检查以下三点，如果都满足则回答 "yes"，否则回答 "no" 并说明原因：
1. Data model 中的字段和类型与 Schema 定义一致
2. Data model 中的字段和类型在不同位置保持一致
3. 如果 Data model 更加合理，可以视为 yes

如果是 no，给出具体修改意见。
"""

print("正在验证 Schema 和 Model 的一致性...")
validation_response = ""
for chunk in llm.llm.stream(validation_prompt):
    validation_response += chunk.content
    print(chunk.content, end="", flush=True)
print()

正在验证 Schema 和 Model 的一致性...
我来逐一检查这三个条件。

## 检查结果：**Yes**

### 详细验证：

**条件1：Data model 中的字段和类型与 Schema 定义一致**

逐表核对关键字段：
- `campus`：所有字段匹配，`map_file_ref` 在 schema 中为字符串但 model 中为 `Optional[str]`，更合理 ✓
- `booking`：`cancel_reason`, `cancelled_at` 为 `Optional` ✓
- `workstation`：`assigned_employee_ref` 为 `Optional[str]` ✓
- `locker`：`assigned_employee_ref`, `assigned_at`, `expires_at` 均为 `Optional` ✓
- `parking_space`：字段完整 ✓
- `shuttle_schedule`：`effective_days: List[EffectiveDay]` 对应 schema 中的数组 ✓
- `canteen_menu`：`dishes: List[DishItem]` 对应 schema 中的数组，且 `DishItem` 子模型字段完整 ✓
- `supply_request`：`items: List[SupplyRequestItem]` 对应 schema 中的数组，子模型字段完整 ✓
- `asset`：`warranty_expires_at` 为 `Optional[str]` ✓
- `repair_request`：无 `zone_ref` 字段，schema 中也无此字段 ✓
- `cleaning_schedule`：`zone_ref`, `assigned_vendor_ref` 均为 `Optional` ✓

**条件2：Data model 中的字段和类型在不同位置保持一致**

- `campus_ref`, `building_ref`, `floor_ref`, `zone_ref` 等外键引用在各模型中类型一致（`str` 或 `Optional[str]`）✓
- `created_at`, `updated_at` 均为 `

### 4. 生成数据脚本（Faker，主流程）

In [ ]:
with open(GENERATED_CODE_DIR / "models.py", 'r', encoding='utf-8') as f:
    model_code = f.read()

# 分析 IT 域的外键依赖（不含 BASE 表），推导生成顺序
ref_map = parse_ref_fields(model_code)
all_tables = list(ref_map.keys())
table_order = topological_sort(all_tables, ref_map)
dependency_info = {
    table: [(field, target) for field, target in refs]
    for table, refs in ref_map.items()
}

print(f"IT 域表生成顺序（拓扑排序，共 {len(table_order)} 张表）：")
for i, t in enumerate(table_order, 1):
    print(f"  {i:2d}. {t}")
print(f"\nFK 依赖映射：")
for t, deps in dependency_info.items():
    if deps:
        print(f"  {t}: {deps}")

with open(PROMPTS_DIR / "3-1_data_script.txt", 'r', encoding='utf-8') as f:
    script_prompt = f.read()

script_prompt = script_prompt.replace("{base_data}", base_data_raw)
script_prompt = script_prompt.replace("{base_gen_data}", base_gen_data_code)
script_prompt = script_prompt.replace("{model_code}", model_code)
script_prompt = script_prompt.replace("{table_order}", str(table_order))
script_prompt = script_prompt.replace("{dependency_info}", str(dependency_info))

local_llm = LLMClient("gpt-5.4-0305-global", 128000, new_config=notebook_config)
print(f"\n正在生成 {DOMAIN} 数据脚本...")
print("-" * 60)
full_response = ""
for chunk in local_llm.llm.stream(script_prompt):
    content = chunk.content
    full_response += content
    print(content, end="", flush=True)
print(f"\n{'-' * 60}")

script_code = extract_code_block(full_response)
ok, err = verify_python_syntax(script_code)
if not ok:
    print(f"❌ 语法错误：{err}")
else:
    print("✓ Python 语法验证通过")

script_path = GENERATED_CODE_DIR / "gen_data.py"
save_generated_file(str(script_path), script_code)
print(f"✓ 数据脚本已保存到: {script_path}")

IT 域表生成顺序（拓扑排序，共 43 张表）：
   1. campuses
   2. shuttlestops
   3. shuttleschedules
   4. shuttlebookings
   5. visitors
   6. badges
   7. supplyrequestitems
   8. supplyrequests
   9. complaints
  10. businesscards
  11. eventregistrations
  12. policies
  13. externalpartners
  14. travelpolicies
  15. cleaningrecords
  16. buildings
  17. parkingspaces
  18. evchargers
  19. shuttleroutes
  20. accesscards
  21. assets
  22. inventoryitems
  23. lostfounds
  24. securityincidents
  25. gifts
  26. servicedirectories
  27. floors
  28. accesslogs
  29. assetrequests
  30. assettransactions
  31. giftrecords
  32. zones
  33. canteens
  34. repairrequests
  35. facilities
  36. workstations
  37. lockers
  38. cleaningschedules
  39. canteenmenus
  40. facilityequipments
  41. bookings
  42. visitorappointments
  43. companyevents

FK 依赖映射：
  campuses: [('campus_ref', 'campuses'), ('map_file_ref', 'map_files')]
  buildings: [('building_ref', 'buildings'), ('campus_ref', 'campuses')]
  

### 3.5 执行数据脚本，生成 db.json

In [17]:
with open(GENERATED_CODE_DIR / "models.py", 'r', encoding='utf-8') as f:
    model_code = f.read()

script_path = GENERATED_CODE_DIR / "gen_data.py"
ref_map = parse_ref_fields(model_code)

print("正在执行数据生成脚本...")
result = subprocess.run(
    [sys.executable, str(script_path)],
    capture_output=True, text=True, timeout=120,
    cwd=str(Path(".").resolve())
)

if result.returncode != 0:
    print(f"❌ 脚本执行失败：\n{result.stderr[-3000:]}")
else:
    print("✓ 脚本执行成功")
    is_json, db_data = verify_json_structure(result.stdout)
    if not is_json:
        print(f"❌ 输出 JSON 解析失败：{db_data}")
    else:
        print(f"✓ JSON 解析成功，共 {len(db_data)} 张表")
        for tname, tdata in db_data.items():
            if isinstance(tdata, dict):
                print(f"  - {tname}: {len(tdata)} 条")

        # FK 完整性验证（ADMIN 域 + BASE 域合并验证）
        merged_data = {**base_data, **db_data}
        fk_ok, fk_errors = validate_foreign_keys(merged_data, ref_map)
        if fk_ok:
            print("✓ 外键完整性验证通过")
        else:
            print(f"⚠️  外键错误 {len(fk_errors)} 处（前5条）：")
            for e in fk_errors[:5]:
                print(f"  - {e}")

        db_path = GENERATED_DATA_DIR / "db.json"
        save_generated_file(str(db_path), result.stdout)
        print(f"\n✓ ADMIN Database 已保存到: {db_path}")

正在执行数据生成脚本...
✓ 脚本执行成功
✓ JSON 解析成功，共 42 张表
  - campus: 10 条
  - building: 27 条
  - floor: 216 条
  - zone: 965 条
  - facility: 599 条
  - facility_equipment: 1538 条
  - booking: 150 条
  - workstation: 4170 条
  - locker: 3709 条
  - parking_space: 200 条
  - ev_charger: 48 条
  - shuttle_route: 20 条
  - shuttle_stop: 96 条
  - shuttle_schedule: 80 条
  - shuttle_booking: 150 条
  - canteen: 20 条
  - canteen_menu: 600 条
  - visitor: 150 条
  - visitor_appointment: 150 条
  - access_card: 125 条
  - access_log: 300 条
  - badge: 75 条
  - asset: 150 条
  - asset_request: 150 条
  - asset_transaction: 44 条
  - inventory_item: 130 条
  - supply_request: 150 条
  - repair_request: 0 条
  - complaint: 150 条
  - lost_found: 150 条
  - security_incident: 150 条
  - gift: 10 条
  - gift_record: 150 条
  - business_card: 57 条
  - company_event: 33 条
  - event_registration: 444 条
  - policy: 10 条
  - service_directory: 8 条
  - external_partner: 25 条
  - travel_policy: 10 条
  - cleaning_schedule: 216 条
  - cleaning_reco

In [18]:
# Pydantic 模型验证：合并 BASE + ADMIN 数据后做完整校验
import sys as _sys
_sys.path.insert(0, str(Path(".").resolve()))
from bench.domains.admin.models import AdminDB

with open(GENERATED_DATA_DIR / "db.json", 'r', encoding='utf-8') as f:
    db_raw = json.load(f)

merged_raw = {**base_data, **db_raw}

try:
    admin_db = AdminDB.model_validate(merged_raw)
    print("✓ AdminDB 模型验证通过")
    print(admin_db.get_statistics())
except Exception as e:
    print(f"❌ 模型验证失败：{e}")

✓ AdminDB 模型验证通过
{'num_users': 75, 'num_employees': 75, 'num_departments': 12, 'num_teams': 22, 'num_roles': 8, 'num_permissions': 29, 'num_campuses': 10, 'num_buildings': 27, 'num_floors': 216, 'num_zones': 965, 'num_facilities': 599, 'num_facility_equipment': 1538, 'num_bookings': 150, 'num_workstations': 4170, 'num_lockers': 3709, 'num_parking_spaces': 200, 'num_ev_chargers': 48, 'num_shuttle_routes': 20, 'num_shuttle_stops': 96, 'num_shuttle_schedules': 80, 'num_shuttle_bookings': 150, 'num_canteens': 20, 'num_canteen_menus': 600, 'num_visitors': 150, 'num_visitor_appointments': 150, 'num_access_cards': 125, 'num_access_logs': 300, 'num_badges': 75, 'num_assets': 150, 'num_asset_requests': 150, 'num_asset_transactions': 44, 'num_inventory_items': 130, 'num_supply_requests': 150, 'num_repair_requests': 0, 'num_complaints': 150, 'num_lost_found': 150, 'num_security_incidents': 150, 'num_gifts': 10, 'num_gift_records': 150, 'num_business_cards': 57, 'num_company_events': 33, 'num_even

### 4. 生成 ADMIN Tools

In [19]:
with open(GENERATED_CODE_DIR / "models.py", 'r', encoding='utf-8') as f:
    model_code = f.read()

with open(PROMPTS_DIR / "4_module_tools.txt", 'r', encoding='utf-8') as f:
    tools_prompt = f.read()

# ADMIN 域功能描述 —— 按数据库表逐条列出，LLM 必须全部实现
admin_function_description = """
## 模块包括的功能范围
```
"行政后勤": [
    {
      "title": "园区地图",
      "definition": "用户希望获取或生成特定园区、办公区域或商业综合体的地理位置信息、建筑布局、功能区域分布、交通路线、设施配置等空间导航和位置服务相关的可视化地图信息。"
    },
    {
      "title": "园区介绍",
      "definition": "用户希望人工智能基于园区的基本信息、发展历程、产业布局、优势特色、招商政策等要素，自动生成结构化的园区宣传介绍文档，用于对外展示园区形象和吸引投资。"
    },
    {
      "title": "会议室查询",
      "definition": "用户希望快速获取会议室的相关信息，包括查询会议室的可用时间、预订状态、容纳人数、设备配置、位置信息，以及进行会议室预订或取消预订等操作。"
    },
    {
      "title": "工位查询",
      "definition": "用户希望查询办公场所内工位的相关信息，包括工位位置、使用状态、预订情况、设备配置、负责人信息或进行工位预约和管理操作。"
    },
    {
      "title": "储物柜查询",
      "definition": "用户希望查询办公场所内储物柜的使用状态、位置信息、租赁情况或申请流程，包括查看可用储物柜、已占用储物柜的使用者信息、储物柜编号位置以及相关的申请和管理操作。"
    },
    {
      "title": "自习室查询",
      "definition": "用户希望查找和预订适合的学习或工作空间，包括查询自习室的位置、可用时间、设施配置、容量限制、预订状态等信息，以便安排个人学习或团队协作活动。"
    },
    {
      "title": "其它设施查询",
      "definition": "用户希望了解办公环境中各类设施设备的信息和状态，包括会议室预订情况、打印机位置及状态、办公用品库存、信息技术设备配置、停车位可用性、食堂菜单、健身房开放时间等办公场所相关设施的查询和预订服务。"
    },
    {
      "title": "食堂信息查询",
      "definition": "用户希望快速获取公司食堂的相关信息，包括今日菜单、营业时间、价格信息、营养成分、特色推荐、就餐人数预估等，以便合理安排用餐计划和时间。"
    },
    {
      "title": "餐补政策",
      "definition": "用户希望了解或查询公司的餐补相关政策制度，包括餐补标准、申请流程、报销规则、适用范围、发放时间等具体规定，以便正确理解和执行餐补相关事务。"
    },
    {
      "title": "商业餐饮查询",
      "definition": "用户希望获取餐饮行业相关的商业信息和数据，包括餐厅推荐、商务用餐场所查询、团建聚餐地点筛选、客户接待餐厅信息、餐饮市场数据分析等，以支持商务活动决策和办公场景中的餐饮需求。"
    },
    {
      "title": "食堂活动",
      "definition": "用户希望人工智能协助策划、组织和管理公司食堂相关的各类活动，包括员工聚餐、节日庆典、团建用餐、美食节等活动的方案制定、流程安排、预算规划和执行跟进。"
    },
    {
      "title": "餐饮投诉",
      "definition": "用户希望人工智能协助处理餐饮相关的投诉事务，包括起草投诉信函、整理投诉要点、分析投诉原因、制定解决方案或跟进投诉处理流程等工作。"
    },
    {
      "title": "超市便利店",
      "definition": "用户希望获取关于零售行业（特别是超市和便利店业态）的经营管理支持，包括商品陈列策略、库存管理方案、促销活动规划、顾客服务流程优化、门店运营数据分析等相关的咨询建议和解决方案。"
    },
    {
      "title": "医务室",
      "definition": "用户希望获得基础的健康咨询、医疗知识普及或办公场所常见健康问题的指导，包括症状初步判断、急救措施建议、职业病预防、健康体检提醒、药物使用咨询等办公场所健康管理相关服务。"
    },
    {
      "title": "自动取款机服务",
      "definition": "用户希望了解或处理与自动取款机相关的业务流程和服务信息，包括自动取款机布点规划、运营状态查询、故障处理流程、交易数据分析、维护计划制定等银行自助设备管理相关的办公需求。"
    },
    {
      "title": "办公用品领用借用",
      "definition": "用户希望通过人工智能助手处理办公用品的申请、审批、登记和归还流程，包括查询可用物品清单、提交领用/借用申请、跟踪审批状态、记录使用情况以及办理归还手续等办公资产管理相关事务。"
    },
    {
      "title": "健身运动设施查询",
      "definition": "用户希望了解公司内部或周边的健身运动设施信息，包括查询健身房位置、开放时间、设备配置、预约方式、使用规则、收费标准或联系方式等，以便合理安排工作之余的运动健身计划。"
    },
    {
      "title": "运动场馆预定",
      "definition": "用户希望通过人工智能助手快速查询、比较和预订各类运动场馆资源，包括会议室、健身房、球场等企业内部或外部场馆，以满足团建活动、体育锻炼、商务会谈等办公相关需求的场地安排。"
    },
    {
      "title": "紧急救助",
      "definition": "用户在遇到紧急工作问题或突发状况时，需要人工智能助手快速提供解决方案、应急处理流程、关键联系人信息或紧急操作指导，以最短时间内恢复正常工作状态或降低损失。"
    },
    {
      "title": "门禁管理",
      "definition": "用户希望通过人工智能助手管理和维护办公场所的门禁系统，包括员工门禁权限的分配与回收、访客临时权限申请与审批、门禁卡的挂失补办、进出记录查询统计，以及门禁设备状态监控等日常管理工作。"
    },
    {
      "title": "工牌管理",
      "definition": "用户希望对员工工牌进行全生命周期管理，包括工牌申请、制作、发放、挂失、补办、回收等流程的跟踪和处理，以及工牌信息的查询、更新和统计分析。"
    },
    {
      "title": "跨园区访问",
      "definition": "用户需要在不同地理位置的办公园区或分支机构之间进行网络连接和资源访问，包括远程访问公司内网系统、跨园区文件共享、视频会议连接，以及解决网络权限、虚拟专用网络配置等技术问题以确保工作连续性。"
    },
    {
      "title": "访客预约",
      "definition": "用户希望通过人工智能助手高效管理和处理访客预约事务，包括预约时间安排、访客信息登记、会议室预订、访问权限申请、预约提醒通知以及访客接待流程的自动化处理。"
    },
    {
      "title": "亲友参观",
      "definition": "用户需要为即将到访的亲朋好友安排参观活动，包括制定参观路线、预约参观时间、准备接待方案、协调相关部门配合、安排讲解人员或准备介绍材料等相关工作事宜。"
    },
    {
      "title": "外部合作",
      "definition": "用户需要与公司外部的合作伙伴、供应商、客户或第三方机构建立、维护或管理合作关系，包括合作伙伴信息查询、合作协议起草、合作项目跟踪、外部沟通协调等相关工作。"
    },
    {
      "title": "园区安全管理",
      "definition": "用户希望通过人工智能助手来协助处理园区安全相关的日常管理工作，包括制定安全管理制度、生成安全检查报告、处理安全事件记录、分析安全风险评估、管理访客登记信息，以及创建安全培训材料等安全管理文档和流程。"
    },
    {
      "title": "失物招领",
      "definition": "用户希望通过人工智能助手快速创建、发布或查询失物招领信息，包括生成失物招领通知模板、整理失物信息清单、匹配失物与寻物信息，以及管理办公场所内的物品遗失与认领流程。"
    },
    {
      "title": "监控查询",
      "definition": "用户希望实时或定期了解业务系统、数据指标、项目进度或关键业务流程的运行状态，包括查看监控面板、获取异常告警、分析趋势变化或生成监控报告。"
    },
    {
      "title": "车位管理",
      "definition": "用户希望通过人工智能助手高效管理停车场或办公区域的车位资源，包括车位分配、预约管理、使用情况统计、费用计算、违规处理等日常运营工作，以提升车位利用率和管理效率。"
    },
    {
      "title": "充电桩查询",
      "definition": "用户希望获取充电桩相关的位置信息、使用状态、收费标准、设备类型等数据，以支持出行规划、设备管理、运营分析或业务决策需求。"
    },
    {
      "title": "班车信息查询",
      "definition": "用户希望快速获取公司班车的相关信息，包括班车路线、发车时间、停靠站点、班车状态、座位预订情况等，以便合理安排通勤出行计划。"
    },
    {
      "title": "园区内交通查询",
      "definition": "用户希望快速获取园区内部的交通信息和出行指导，包括查询园区内班车时刻表、停车位信息、各建筑物之间的路线规划、园区地图导航以及相关交通设施的位置和使用说明。"
    },
    {
      "title": "差旅酒店咨询",
      "definition": "用户希望获取商务出行相关的酒店信息和预订建议，包括查询目的地酒店推荐、价格对比、位置便利性分析、企业协议酒店查询、差旅政策咨询以及预订流程指导等，以提高出差效率和成本控制。"
    },
    {
      "title": "差旅交通咨询",
      "definition": "用户希望获取出行相关的信息和建议，包括查询航班、火车、酒店等交通住宿信息，了解目的地天气、交通路线、费用预算，以及获取差旅政策、报销流程等相关指导。"
    },
    {
      "title": "差旅餐标咨询",
      "definition": "用户希望了解和查询公司差旅出行相关的政策标准和报销规定，包括住宿标准、餐饮补贴、交通费用限额、不同城市或职级的差旅标准差异，以及相关审批流程和报销要求。"
    },
    {
      "title": "其它差旅咨询",
      "definition": "用户希望获得与商务出行相关的信息和建议，包括查询航班、酒店、交通方式、目的地天气、签证要求、差旅政策、费用预算、行程规划等，以便高效完成出差安排和提升出行体验。"
    },
    {
      "title": "接待标准",
      "definition": "用户希望了解或制定在特定场景下的服务规范、流程标准或质量要求，包括客户接待流程、会议接待标准、访客接待规范等标准化服务指引的查询或制定。"
    },
    {
      "title": "礼品管理",
      "definition": "用户希望系统化地管理企业礼品资源，包括礼品库存查询、采购计划制定、发放记录跟踪、预算控制以及礼品分类管理等，以提高礼品使用效率和成本控制能力。"
    },
    {
      "title": "设施报修",
      "definition": "用户希望通过人工智能助手快速提交、跟踪和管理办公场所内各类设施设备的故障报修申请，包括填写报修单据、查询维修进度、联系相关维修人员或获取设施维护相关信息。"
    },
    {
      "title": "环境打扫维护",
      "definition": "用户希望获得办公环境清洁和设施维护方面的指导和管理支持，包括制定清洁计划、维护设备保养提醒、环境卫生标准制定、清洁用品采购建议、维修服务安排等，以确保办公场所的整洁有序和设备正常运行。"
    },
    {
      "title": "行政物品领用借用",
      "definition": "用户希望通过人工智能助手处理办公用品、设备或其他行政资源的申请、审批、登记和归还流程，包括查询物品库存状态、提交领用借用申请、跟踪审批进度、生成领用单据以及管理归还提醒等行政管理事务。"
    },
    {
      "title": "项目室申请",
      "definition": "用户需要为特定项目或团队申请独立的工作空间，包括申请项目室的创建、配置项目成员权限、设置项目资源访问范围，以及管理项目室的生命周期和相关审批流程。"
    },
    {
      "title": "名片制作",
      "definition": "用户希望人工智能协助创建个人或企业名片，包括设计布局、内容编排、联系信息整理等，以生成符合商务场景需求的专业名片。"
    },
    {
      "title": "美餐服务",
      "definition": "用户希望获得餐饮相关的服务支持，包括团餐预订、菜单推荐、营养搭配建议、用餐时间安排、餐厅信息查询等办公场景下的用餐需求解决方案。"
    },
    {
      "title": "公司活动日",
      "definition": "用户希望了解和查询公司或者集团的重要日程安排、关键时间节点、企业文化活动或历史性事件信息，以便更好地规划工作安排和参与相关活动。"
    },
    {
      "title": "行政服务台查询",
      "definition": "用户希望快速获取公司内部行政相关的信息和服务，包括查询办公用品申请流程、会议室预订状态、员工福利政策、报销规定、人事制度、信息技术设备申请、访客登记要求等行政事务的具体信息和办理方式。"
    },
    {
      "title": "行政热线查询",
      "definition": "用户希望快速获取公司内部行政相关的联系方式和服务信息，包括各部门热线电话、服务时间、业务范围、紧急联系人或特定行政事务的处理流程和负责人信息。"
    },
    {
      "title": "资产申请与领用",
      "definition": "用户的核心目的是申请获取公司内部各类资产的使用权限，或办理已申请资产的领用手续。这包括信息技术设备申请、办公用品领用、会议室预定、软件许可申请、固定资产借用等流程的发起、审批状态查询和相关表单填写。"
    },
    {
      "title": "资产借用",
      "definition": "用户需要临时获取或使用公司内部的各种资源和设备，包括会议室预订、办公设备申请、车辆借用、信息技术设备调配等固定资产或共享资源的申请和管理流程。"
    },
    {
      "title": "资产更换",
      "definition": "用户需要对已有办公或信息化资产进行更换，包括因设备损坏、老化、丢失、性能不足或岗位调整而申请替换电脑、显示器、手机、工牌等资产，并涉及旧资产回收、新资产发放、信息登记及审批流转等相关流程。"
    },
    {
      "title": "资产归还",
      "definition": "用户需要处理公司资产的归还流程，包括记录归还的设备、物品或资源信息，生成归还清单，跟踪归还状态，以及完成相关的审批和确认手续。"
    },
    {
      "title": "资产自购",
      "definition": "用户需要为公司或部门采购各类资产设备，包括办公用品、信息技术设备、软件许可、办公家具等的需求申请、供应商比较、采购流程跟踪、预算审批及资产入库管理等全流程支持。"
    },
    {
      "title": "资产损坏与维修",
      "definition": "用户希望记录、跟踪和管理公司固定资产的损坏情况及维修流程，包括损坏报告提交、维修申请审批、维修进度跟踪、维修成本统计以及资产状态更新等办公管理需求。"
    },
    {
      "title": "资产查询",
      "definition": "用户希望了解和检索公司内部各类主题相关的资产信息，包括查询特定主题下的文档、模板、案例、最佳实践等资源的存在性、内容概要、使用场景、创建时间、负责人员以及资产质量评估等信息。"
    },
    {
      "title": "会议室预订",
      "definition": "用户希望通过人工智能助手快速查询会议室的可用性、预订合适的会议室资源，或管理已有的会议室预订信息，包括查看预订状态、修改预订时间、取消预订等会议室相关的办公需求。"
    },
    {
      "title": "会议室设备管理",
      "definition": "用户希望高效管理和维护办公环境中的会议室设备资源，包括查询设备状态、预约设备使用、报告设备故障、跟踪设备维修进度、统计设备使用率以及优化设备配置等操作。"
    }
  ]
```
"""

with open(BASE_CODE_DIR / "tools.py", 'r', encoding='utf-8') as f:
    base_tools_code = f.read()

tools_prompt = tools_prompt.replace("{domain}", DOMAIN)
tools_prompt = tools_prompt.replace("{base_tools}", base_tools_code)
tools_prompt = tools_prompt.replace("{model_code}", model_code)
tools_prompt = tools_prompt.replace("{function_description}", admin_function_description)

print(f"正在生成 {DOMAIN} Tools...")
print("-" * 60)
tools_response = ""
for chunk in llm.llm.stream(tools_prompt):
    content = chunk.content
    tools_response += content
    print(content, end="", flush=True)
print(f"\n{'-' * 60}")

tools_code = extract_code_block(tools_response)
is_valid, error = verify_python_syntax(tools_code)
if is_valid:
    print("✓ Python 语法验证通过")
else:
    print(f"✗ Python 语法错误: {error}")

tools_path = GENERATED_CODE_DIR / "tools.py"
save_generated_file(str(tools_path), tools_code)
print(f"✓ Tools 已保存到: {tools_path}")

正在生成 admin Tools...
------------------------------------------------------------
```python
"""Toolkit for the ADMIN domain, extending OfficeTools with administrative operations."""
from typing import Dict, List, Optional
import json
from datetime import datetime, timezone

from bench.environment.toolkit import ToolType, is_tool
from bench.domains.base.tools import OfficeTools
from bench.domains.admin.models import (
    AdminDB,
    Campus,
    Building,
    Floor,
    Zone,
    Facility,
    FacilityEquipment,
    Booking,
    Workstation,
    Locker,
    ParkingSpace,
    EvCharger,
    ShuttleRoute,
    ShuttleStop,
    ShuttleSchedule,
    ShuttleBooking,
    Canteen,
    CanteenMenu,
    Visitor,
    VisitorAppointment,
    AccessCard,
    AccessLog,
    Badge,
    Asset,
    AssetRequest,
    AssetTransaction,
    InventoryItem,
    SupplyRequest,
    SupplyRequestItem,
    RepairRequest,
    Complaint,
    LostFound,
    SecurityIncident,
    Gift,
    GiftRecord,
    BusinessCa